# TreeLongestZigZag

[Codility task](https://app.codility.com/programmers/trainings/7/tree_longest_zig_zag/) - Exercise 7, Data Structures - *medium*

A **turn** is a change of direction (left -> right or right -> left) on a downward path of a binary tree. The length of a zigzag is its number of turns. A path with a single edge (or node) has length `0`.

`solution(T)` returns the maximal number of turns on a downward path **starting at the root**.

```
(5, (3, (20, (6, None, None), None), None),
    (10, (1, None, None),
         (15, (30, None, (9, None, None)), (8, None, None))))
```

The path `5 -> 10 -> 15 -> 30 -> 9` (R, R, L, R) has turns at `15` and `30`: answer `2`.

Assumptions: `N` in `[1..100,000]`, height in `[0..800]`.

In [ ]:
from dataclasses import dataclass


@dataclass
class Tree:
    x: int = 0
    l: "Tree" = None
    r: "Tree" = None


def from_tuple(t):
    """Build a Tree from the (X, L, R) notation used in the statement (None = empty)."""
    if t is None:
        return None
    x, l, r = t
    return Tree(x, from_tuple(l), from_tuple(r))

## Idea: two values per node

For every node store the best number of turns obtainable going **downwards**, given the direction of the edge that arrived at the node:

- `arrived_left(node)`: continue left (no turn) -> `arrived_left(l)`; or go right (a turn) -> `1 + arrived_right(r)`;
- `arrived_right(node)`: continue right -> `arrived_right(r)`; or go left (a turn) -> `1 + arrived_left(l)`;
- a leaf has `(0, 0)`.

At the root there is no previous direction, so the first edge is never a turn: the answer is `max(arrived_left(l), arrived_right(r))` (or `0` for a single node).

The values of a node depend on its children, so we use an **iterative post-order** traversal (explicit stack) instead of recursion. Time O(N), space O(N).

In [ ]:
def solution(T):
    best = {}                       # id(node) -> (arrived_left, arrived_right)
    stack = [(T, False)]
    while stack:
        node, children_done = stack.pop()
        if not children_done:
            stack.append((node, True))
            if node.l is not None:
                stack.append((node.l, False))
            if node.r is not None:
                stack.append((node.r, False))
            continue

        arrived_left = arrived_right = 0
        if node.l is not None:
            l_left, _ = best[id(node.l)]
            arrived_left = max(arrived_left, l_left)            # keep going left
            arrived_right = max(arrived_right, 1 + l_left)      # turn: came from the right, go left
        if node.r is not None:
            _, r_right = best[id(node.r)]
            arrived_right = max(arrived_right, r_right)         # keep going right
            arrived_left = max(arrived_left, 1 + r_right)       # turn: came from the left, go right
        best[id(node)] = (arrived_left, arrived_right)

    # The first edge from the root never counts as a turn.
    answer = 0
    if T.l is not None:
        answer = max(answer, best[id(T.l)][0])
    if T.r is not None:
        answer = max(answer, best[id(T.r)][1])
    return answer

## Official example

In [ ]:
example = from_tuple(
    (5, (3, (20, (6, None, None), None), None),
        (10, (1, None, None),
             (15, (30, None, (9, None, None)), (8, None, None))))
)
assert solution(example) == 2
print("example OK")

## Edge cases

In [ ]:
assert solution(from_tuple((1, None, None))) == 0                       # single node
assert solution(from_tuple((1, (2, None, None), None))) == 0            # one edge
assert solution(from_tuple((1, (2, (3, None, None), None), None))) == 0 # straight left
assert solution(from_tuple((1, None, (2, None, (3, None, None))))) == 0 # straight right
assert solution(from_tuple((1, (2, None, (3, None, None)), None))) == 1 # L then R: one turn
assert solution(from_tuple((1, None, (2, (3, None, None), None)))) == 1 # R then L: one turn
assert solution(from_tuple((1, (2, None, (3, (4, None, None), None)), None))) == 2   # L R L
print("edge cases OK")

## Brute-force check

The reference enumerates **every** downward path from the root with plain recursion, counting the direction changes. Random trees are built by attaching new nodes to random free child slots.

In [ ]:
import random


def reference(T):
    best = 0

    def walk(node, last_dir, turns):
        nonlocal best
        best = max(best, turns)
        for d, child in (("L", node.l), ("R", node.r)):
            if child is not None:
                walk(child, d, turns + (last_dir is not None and d != last_dir))

    walk(T, None, 0)
    return best


def random_tree(n, rng):
    root = Tree(0)
    free = [(root, "l"), (root, "r")]
    for i in range(1, n):
        parent, side = free.pop(rng.randrange(len(free)))
        node = Tree(i)
        setattr(parent, side, node)
        free += [(node, "l"), (node, "r")]
    return root


rng = random.Random(11)
for _ in range(3000):
    tree = random_tree(rng.randint(1, 60), rng)
    assert solution(tree) == reference(tree)
print("matches the reference on 3000 random trees")

## Large and extreme trees

- a zigzag chain of height 800: the first edge is free, so `799` turns;
- a straight chain: `0` turns;
- a complete binary tree with 65,535 nodes;
- a random tree with 100,000 nodes (its height is not limited here, which is why the solution is iterative).

In [ ]:
import time


def chain(height, zigzag):
    root = cur = Tree(0)
    side = "l"
    for _ in range(height):
        child = Tree(0)
        setattr(cur, side, child)
        cur = child
        if zigzag:
            side = "r" if side == "l" else "l"
    return root


def complete(depth):
    if depth == 0:
        return None
    return Tree(0, complete(depth - 1), complete(depth - 1))


assert solution(chain(800, zigzag=True)) == 799
assert solution(chain(800, zigzag=False)) == 0

tree = complete(16)                 # 65,535 nodes, height 15
assert solution(tree) == reference(tree) == 14

rng = random.Random(2)
big = random_tree(100_000, rng)
start = time.perf_counter()
result = solution(big)
print(f"random tree with 100,000 nodes: answer {result} in {time.perf_counter() - start:.3f}s")

## Notes

- The first edge from the root never counts: the root has no incoming direction.
- Nodes are not hashable (`@dataclass` with `eq=True` sets `__hash__` to `None`), so the dictionary is keyed by `id(node)`.
- Never compare or print big trees: the generated `__eq__` and `__repr__` are recursive.
- An equivalent top-down formulation carries `(node, last_direction, turns)` on the stack and keeps the maximum; the bottom-up version above avoids storing the path state per node.